# 🎙️ Qwen3-TTS Voice Cloning on Google Colab

Run **Qwen3-TTS 1.7B Base** for high-quality voice cloning directly on Google Colab.

## What you will do
1. Check your GPU
2. Install dependencies
3. Restart Colab once
4. Verify the environment
5. Load the 1.7B model
6. Create the voice-cloning function
7. Launch the Gradio interface

### Recommended runtime
**Runtime → Change runtime type → GPU**

A T4 GPU or better is recommended. Run every step in order.


## Step 1 — Check your GPU

This confirms that Colab has assigned a GPU to your session.

In [ ]:
!nvidia-smi

## Step 2 — Install required dependencies

This cell:
- Installs **SoX** for audio processing
- Keeps `setuptools` compatible with Colab's PyTorch
- Installs Qwen3-TTS, Gradio, and audio libraries

**FlashAttention is intentionally not installed.** It is optional and can cause dependency/build problems on changing Colab environments.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq sox libsox-fmt-all
!pip install -q --upgrade "setuptools<82" wheel
!pip install -q --upgrade qwen-tts gradio soundfile librosa
print("✅ Installation complete!")
print("➡️ Run Step 3 to restart the runtime.")

## Step 3 — Restart the runtime

The newly installed packages need a clean Python restart.

After Colab reconnects, **do not run this cell again**. Continue from Step 4.


In [ ]:
import os, signal
print("Restarting runtime...")
os.kill(os.getpid(), signal.SIGKILL)

# After the restart, continue here ⬇️

## Step 4 — Verify the environment

Checks GPU, CUDA, PyTorch, and available VRAM.

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ No GPU detected. Enable GPU in Runtime → Change runtime type."
print("✅ GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

## Step 5 — Load Qwen3-TTS 1.7B

Loads **`Qwen/Qwen3-TTS-12Hz-1.7B-Base`**, the larger model for better voice-cloning quality.

First run downloads several GB.

### Common warnings
- `HF_TOKEN` missing: safe for this public model
- `flash-attn` missing: safe; only affects speed


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tts = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=DTYPE,
)

print("✅ Model loaded successfully!")
print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))

## Step 6 — Create the voice-cloning function

This function:
1. Receives your reference audio
2. Uses the reference transcript when available
3. Generates new speech in the cloned voice
4. Saves the result as a WAV file

**Best quality:** provide an exact transcript of the reference audio.


In [ ]:
import os, tempfile, torch
import soundfile as sf
import gradio as gr

def clone_voice(reference_audio, reference_text, target_text, language):
    if reference_audio is None:
        raise gr.Error("Please upload a reference audio file.")
    if not target_text or not target_text.strip():
        raise gr.Error("Please enter the text you want to generate.")

    ref_text = reference_text.strip() if reference_text and reference_text.strip() else None

    with torch.inference_mode():
        wavs, sr = tts.generate_voice_clone(
            text=target_text.strip(),
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_text,
            x_vector_only_mode=(ref_text is None),
            do_sample=True,
        )

    output_path = os.path.join(tempfile.gettempdir(), "qwen3_tts_cloned_voice.wav")
    sf.write(output_path, wavs[0], sr)
    return output_path

print("✅ Voice cloning function is ready!")

## Step 7 — Launch the Web Interface

The interface lets you:
- Upload a reference voice
- Add the reference transcript
- Choose the target language
- Enter new text
- Generate cloned speech

### Best results
Use **10–30 seconds** of clean speech, one speaker, minimal noise, and an exact transcript.


In [ ]:
languages = [
    "Auto", "English", "Chinese", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese", "Spanish", "Italian"
]

with gr.Blocks(title="Qwen3-TTS Voice Cloning") as demo:
    gr.Markdown("# 🎙️ Qwen3-TTS Voice Cloning")
    gr.Markdown("Upload a reference voice and generate new speech.")

    with gr.Row():
        with gr.Column():
            reference_audio = gr.Audio(label="1. Upload Reference Audio", type="filepath")
            reference_text = gr.Textbox(
                label="2. Reference Transcript (Recommended)",
                placeholder="Type exactly what is spoken in the reference audio...",
                lines=3
            )
            language = gr.Dropdown(languages, value="Auto", label="3. Target Language")

        with gr.Column():
            target_text = gr.Textbox(
                label="4. Text to Generate",
                placeholder="Type what you want the cloned voice to say...",
                lines=6
            )
            generate_btn = gr.Button("🎙️ Generate Cloned Voice", variant="primary")
            output_audio = gr.Audio(label="Generated Audio", type="filepath")

    generate_btn.click(
        clone_voice,
        inputs=[reference_audio, reference_text, target_text, language],
        outputs=output_audio,
    )

print("🚀 Starting Qwen3-TTS...")
demo.launch(share=True, debug=True)

# 🎉 Done!

## How to use
1. Upload reference audio
2. Enter the exact words spoken in it
3. Select a language or use Auto
4. Enter the new text
5. Click **Generate Cloned Voice**

## Quality tips
- Use clean audio without music
- Use 10–30 seconds of speech
- Provide an exact transcript
- Use only one speaker

## Warning guide
**FlashAttention missing:** not an error; only affects speed.  
**HF_TOKEN missing:** not an error for this public model.  
**Model loaded successfully:** everything is working correctly.

⚠️ Only clone voices you have permission to use.
